# EDL+ORCU Fluorosis Diagnosis — Kaggle Training Notebook

**Tasks**: Dental Fluorosis (DF, ViT-Base) + Skeletal Fluorosis (SF, ResNet-50), 4-class ordinal classification.

**5-mode comparison**: CE | Cumulative (Coral) | SORD | EDL | EDL+ORCU

**Code**: Cloned from GitHub (`XiaoHG/fluorosis_project`) during setup.
**Expected runtime**: ~5-6 hours on T4 x2 GPU.

**Version**: v2.2 (2026-05-16) — Fix build_model mode dispatch (CE/SORD → StandardClassifier)

**Bug fixes from v2.1**:
- CE/SORD now use StandardClassifier (fc+dropout, no EDL head) instead of EDLClassifier
- Previous bug: all modes except cumulative shared same EDLClassifier → CE and EDL+ORCU produced identical results
- CE logger: `L_ce: 0.0` → `L_ce: loss.item()` (logging only)

## 1. Environment Setup

In [ ]:
!pip install transformers scikit-learn openpyxl scipy -q

# Clone code from GitHub (internet available only during setup)
!rm -rf /kaggle/working/fluorosis_project
!git clone https://github.com/XiaoHG/fluorosis_project.git /kaggle/working/fluorosis_project

# Verify clone succeeded
import os
assert os.path.isdir("/kaggle/working/fluorosis_project/06_Implementation"), \
    "Git clone failed! Is the repo public? If private, use: https://<token>@github.com/XiaoHG/fluorosis_project.git"
print("Code cloned successfully.")

In [ ]:
# ---- Master Setup: run this cell first ----
import os, sys, json, copy
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from scipy import stats

# Pre-download pretrained weights while internet is available
print("Downloading pretrained weights (internet available only during setup)...")
from transformers import ViTModel
_ = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
import torchvision.models as tv_models
_ = tv_models.resnet50(weights="IMAGENET1K_V2")
print("Weights cached successfully.")

# Paths
CODE_ROOT = "/kaggle/working/fluorosis_project/06_Implementation"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
OUTPUT_DIR = "/kaggle/working"
sys.path.insert(0, CODE_ROOT)

from data import create_dataloaders, DFDataset, SFDataset, get_transforms
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
print(f"Device: {device} | GPU: {gpu_name}")
print(f"CUDA available: {torch.cuda.is_available()} | GPU count: {torch.cuda.device_count()}")
print("Setup complete. You can now run any subsequent cell.")

## 2. Data Verification
Check that both DF and SF datasets are accessible with correct class distributions.

In [ ]:
def verify_data(task):
    """Verify dataset structure and print class distribution per split."""
    import sys, os
    import numpy as np
    sys.path.insert(0, "/kaggle/working/fluorosis_project/06_Implementation")
    from data import DFDataset, SFDataset
    DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
    ds_cls = DFDataset if task == "df" else SFDataset
    for split in ["train", "val", "test"]:
        ds = ds_cls(DATA_ROOT, split=split)
        labels = np.array([ds[i][1] for i in range(len(ds))])
        dist = {f"class_{k}": int((labels == k).sum()) for k in range(4)}
        print(f"  {task.upper()} {split:5s}: {len(ds):3d} samples | {dist}")

print("=== Data Verification ===")
verify_data("df")
print()
verify_data("sf")
print("\nData OK.")

## 3. Training Functions
Self-contained training loop. Supports 5 modes: `ce`, `cumulative`, `sord`, `edl`, `edl_orcu`. Default orcu_lambda_reg=0.01.

In [ ]:
# ---- Mode-specific loss wrappers (self-contained) ----
import torch.nn as nn
import torch.nn.functional as F


class CEWrapper(nn.Module):
    """Standard cross-entropy (one-hot targets)."""
    def forward(self, alpha, z, targets, epoch=None):
        loss = F.nll_loss(F.log_softmax(z, dim=-1), targets)
        return loss, {"stage": 0, "L_ce": loss.item()}


class CumulativeWrapper(nn.Module):
    """Coral/CORN: K-1 binary cross-entropy. Requires model.ordinal_logits."""
    def __init__(self, num_classes=4):
        super().__init__()
        from losses.cumulative_loss import CumulativeLoss
        self.cum_loss = CumulativeLoss(num_classes=num_classes)
        self.model_ref = None

    def set_model(self, model):
        self.model_ref = model

    def forward(self, alpha, z, targets, epoch=None):
        if self.model_ref is not None and hasattr(self.model_ref, 'ordinal_logits'):
            ol = self.model_ref.ordinal_logits
        else:
            ol = z
        loss = self.cum_loss(ol, targets)
        return loss, {"stage": 0, "L_cum": loss.item()}


class SORDWrapper(nn.Module):
    """SORD soft-encoded CE (no ordinal regularization)."""
    def __init__(self, num_classes=4):
        super().__init__()
        from losses.orcu_loss import ORCULoss
        self.sord = ORCULoss(num_classes=num_classes, t=3.0, lambda_reg=0.0)

    def forward(self, alpha, z, targets, epoch=None):
        loss = self.sord(z, targets)
        return loss, {"stage": 0, "L_sord": loss.item()}


class EDLWrapper(nn.Module):
    """EDL only (one-hot targets + KL annealing)."""
    def __init__(self, num_classes=4, kl_lambda=0.1, total_epochs=100):
        super().__init__()
        from losses.edl_loss import EDLLoss
        self.edl = EDLLoss(num_classes=num_classes, kl_lambda=kl_lambda)
        self.total_epochs = total_epochs

    def forward(self, alpha, z, targets, epoch=None):
        loss = self.edl(alpha, targets, epoch, self.total_epochs)
        return loss, {"stage": 0, "L_edl": loss.item()}


print("Loss wrappers defined: CE, Cumulative, SORD, EDL, EDL+ORCU")

In [ ]:
# ---- Optimizer & Scheduler builders ----
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR


def build_optimizer(model, lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.05):
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("backbone"):
            backbone_params.append(p)
        else:
            head_params.append(p)
    return optim.AdamW([
        {"params": backbone_params, "lr": lr_backbone},
        {"params": head_params, "lr": lr_head},
    ], weight_decay=weight_decay)


def build_scheduler(optimizer, epochs, warmup_epochs):
    warmup_sched = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine_sched = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
    return SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_epochs])

print("Optimizer/scheduler builders ready.")

In [ ]:
# ---- Core training function (fully self-contained) ----
import os, sys, json, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

sys.path.insert(0, "/kaggle/working/fluorosis_project/06_Implementation")
from data import create_dataloaders
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


@torch.no_grad()
def evaluate_model(model, dataloader):
    """Run full evaluation over a dataloader."""
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None:
            all_alpha.append(alpha.cpu())
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha, dim=0) if all_alpha else None
    z = torch.cat(all_z, dim=0)
    targets = torch.cat(all_targets, dim=0)
    return compute_metrics(alpha, z, targets)


def train_model(task, data_root, output_dir, mode="edl_orcu",
                batch_size=None, epochs=None, lr_backbone=1e-4, lr_head=1e-3,
                stage_1_epochs=None, stage_2_epochs=None,
                lambda_orcu=0.5, lambda_kl=0.1, orcu_t=3.0, lambda_reg=0.01,
                warmup_epochs=None, weight_decay=0.05, seed=42):
    """Train one model configuration. Returns (test_metrics, history)."""
    # Task defaults
    if task == "df":
        batch_size = batch_size or 32
        epochs = epochs or 100
        stage_1_epochs = stage_1_epochs or 5
        stage_2_epochs = stage_2_epochs or 30
        warmup_epochs = warmup_epochs or 5
    else:
        batch_size = batch_size or 16
        epochs = epochs or 150
        stage_1_epochs = stage_1_epochs or 10
        stage_2_epochs = stage_2_epochs or 45
        warmup_epochs = warmup_epochs or 10

    torch.manual_seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    # Data
    train_loader, val_loader, test_loader = create_dataloaders(
        data_root, task=task, batch_size=batch_size, num_workers=2)
    print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} "
          f"test={len(test_loader.dataset)}")

    # Model — build_model now dispatches: ce/sord→StandardClassifier, cumulative→CumulativeClassifier, edl/edl_orcu→EDLClassifier
    model = build_model(task=task, mode=mode)
    model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    model_type = type(model).__name__
    print(f"Model: {n_params:,} params | Mode: {mode} | Type: {model_type}")

    # Criterion
    if mode == "ce":
        class CELocal(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                loss = F.nll_loss(F.log_softmax(z, dim=-1), targets)
                return loss, {"stage": 0, "L_ce": loss.item()}
        criterion = CELocal()

    elif mode == "cumulative":
        from losses.cumulative_loss import CumulativeLoss
        cum_fn = CumulativeLoss(num_classes=4)
        class CumLocal(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                ol = model.ordinal_logits if hasattr(model, 'ordinal_logits') else z
                loss = cum_fn(ol, targets)
                return loss, {"stage": 0, "L_cum": loss.item()}
        criterion = CumLocal()

    elif mode == "sord":
        from losses.orcu_loss import ORCULoss
        sord_fn = ORCULoss(num_classes=4, t=orcu_t, lambda_reg=0.0)
        class SORDLocal(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                loss = sord_fn(z, targets)
                return loss, {"stage": 0, "L_sord": loss.item()}
        criterion = SORDLocal()

    elif mode == "edl":
        from losses.edl_loss import EDLLoss
        edl_fn = EDLLoss(num_classes=4, kl_lambda=lambda_kl)
        class EDLLocal(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                loss = edl_fn(alpha, targets, epoch, epochs)
                return loss, {"stage": 0, "L_edl": loss.item()}
        criterion = EDLLocal()

    else:  # edl_orcu
        criterion = CombinedLoss(
            num_classes=4, lambda_orcu=lambda_orcu, lambda_kl=lambda_kl, orcu_t=orcu_t,
            orcu_lambda_reg=lambda_reg,
            stage_1_epochs=stage_1_epochs, stage_2_epochs=stage_2_epochs, total_epochs=epochs)

    # Optimizer & scheduler
    def build_opt(m):
        bb, hd = [], []
        for n, p in m.named_parameters():
            if not p.requires_grad:
                continue
            (bb if n.startswith("backbone") else hd).append(p)
        return optim.AdamW([{"params": bb, "lr": lr_backbone}, {"params": hd, "lr": lr_head}],
                           weight_decay=weight_decay)

    def build_sched(opt, ep, wu):
        wu_s = LinearLR(opt, start_factor=0.1, total_iters=wu)
        cos = CosineAnnealingLR(opt, T_max=ep - wu)
        return SequentialLR(opt, schedulers=[wu_s, cos], milestones=[wu])

    optimizer = build_opt(model)
    scheduler = build_sched(optimizer, epochs, warmup_epochs)

    # Training loop
    best_val_acc, best_state = 0.0, None
    history = []

    for epoch in range(epochs):
        model.train()
        epoch_losses = {"L_ce": 0.0, "L_cum": 0.0, "L_sord": 0.0,
                        "L_edl": 0.0, "L_orcu": 0.0, "L_total": 0.0}
        epoch_stage, n_batches = 0, 0

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, loss_info = criterion(alpha, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_stage = loss_info.get("stage", 0)
            for k in epoch_losses:
                if k in loss_info:
                    epoch_losses[k] += loss_info[k]
            n_batches += 1

        scheduler.step()
        for k in epoch_losses:
            epoch_losses[k] /= max(n_batches, 1)

        val_metrics = evaluate_model(model, val_loader)
        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            best_state = copy.deepcopy(model.state_dict())

        history.append({
            "epoch": epoch, "stage": epoch_stage,
            "train_loss": epoch_losses,
            "val_metrics": {k: v for k, v in val_metrics.items()
                            if isinstance(v, (float, int)) and not k.startswith("evidence_class_")},
        })

        loss_key = next((k for k in ["L_total", "L_ce", "L_cum", "L_sord", "L_edl", "L_orcu"]
                         if epoch_losses.get(k, 0) != 0), "L_total")
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == epochs - 1:
            print(f"[E{epoch+1:3d}/{epochs}] S={epoch_stage} | {loss_key}={epoch_losses.get(loss_key, 0):.4f} | "
                  f"Acc={val_metrics['acc']:.4f} F1={val_metrics['macro_f1']:.4f} "
                  f"QWK={val_metrics['qwk']:.4f} ECE={val_metrics['ece']:.4f} "
                  f"Unim={val_metrics['pct_unimodal']:.2%} U={val_metrics['mean_uncertainty']:.3f}")

    # Load best and test
    model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_loader)

    torch.save({"model_state_dict": best_state, "test_metrics": test_metrics},
               os.path.join(output_dir, "best.pt"))
    with open(os.path.join(output_dir, "test_metrics.json"), "w") as f:
        json.dump({k: v for k, v in test_metrics.items() if isinstance(v, (float, int))}, f, indent=2)
    with open(os.path.join(output_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print(f"\n[Test] Acc={test_metrics['acc']:.4f} F1={test_metrics['macro_f1']:.4f} "
          f"QWK={test_metrics['qwk']:.4f} ECE={test_metrics['ece']:.4f} "
          f"Unimodal={test_metrics['pct_unimodal']:.2%} U-ECE={test_metrics['u_ece']:.4f} "
          f"AUROC(u)={test_metrics['auroc_u']:.4f}")
    return test_metrics, history

print("train_model() and evaluate_model() ready.")

## 4. Skeletal Fluorosis (SF) Experiments
ResNet-50 backbone, 5 modes: CE, Cumulative, SORD, EDL, EDL+ORCU.

In [ ]:
import os, json
OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"

sf_results = {}
sf_modes = ["ce", "cumulative", "sord", "edl", "edl_orcu"]

for mode in sf_modes:
    print(f"\n{'='*60}")
    print(f"SF | Mode: {mode}")
    print(f"{'='*60}")
    out_dir = os.path.join(OUTPUT_DIR, f"sf_{mode}")
    metrics, _ = train_model("sf", DATA_ROOT, out_dir, mode=mode, epochs=150, batch_size=16)
    sf_results[mode] = {k: v for k, v in metrics.items() if isinstance(v, (float, int))}

# Save
with open(os.path.join(OUTPUT_DIR, "sf_all_results.json"), "w") as f:
    json.dump(sf_results, f, indent=2)

print("\n=== SF Results ===")
print(f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
print("-" * 78)
for mode, m in sf_results.items():
    print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} {m.get('qwk',0):8.4f} "
          f"{m.get('ece',0):8.4f} {m.get('u_ece',0):8.4f} {m.get('auroc_u',0):8.4f} "
          f"{m.get('pct_unimodal',0):7.2%}")

## 5. Dental Fluorosis (DF) Experiments
ViT-Base backbone, 5 modes: CE, Cumulative, SORD, EDL, EDL+ORCU.

In [ ]:
import os, json
OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"

df_results = {}
df_modes = ["ce", "cumulative", "sord", "edl", "edl_orcu"]

for mode in df_modes:
    print(f"\n{'='*60}")
    print(f"DF | Mode: {mode}")
    print(f"{'='*60}")
    out_dir = os.path.join(OUTPUT_DIR, f"df_{mode}")
    metrics, _ = train_model("df", DATA_ROOT, out_dir, mode=mode, epochs=100, batch_size=32)
    df_results[mode] = {k: v for k, v in metrics.items() if isinstance(v, (float, int))}

# Save
with open(os.path.join(OUTPUT_DIR, "df_all_results.json"), "w") as f:
    json.dump(df_results, f, indent=2)

print("\n=== DF Results ===")
print(f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
print("-" * 78)
for mode, m in df_results.items():
    print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} {m.get('qwk',0):8.4f} "
          f"{m.get('ece',0):8.4f} {m.get('u_ece',0):8.4f} {m.get('auroc_u',0):8.4f} "
          f"{m.get('pct_unimodal',0):7.2%}")

## 6. 5-Fold Cross-Validation
Stratified K-fold CV: CE, Cumulative, EDL, EDL+ORCU on both tasks. Paired t-tests between all method pairs.

In [ ]:
# ---- 5-Fold Cross-Validation (self-contained) ----
import os, sys, json, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from scipy import stats

sys.path.insert(0, "/kaggle/working/fluorosis_project/06_Implementation")
from data import DFDataset, SFDataset, get_transforms
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def run_cv(task, data_root, methods=None, n_folds=5, epochs=80, seed=42):
    if methods is None:
        methods = ["ce", "cumulative", "edl", "edl_orcu"]

    if task == "df":
        label_ds = DFDataset(data_root, split="train", split_seed=seed)
    else:
        label_ds = SFDataset(data_root, split="train")
    labels = np.array([label_ds[i][1] for i in range(len(label_ds))])

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold_results = {m: [] for m in methods}
    batch_size = 32 if task == "df" else 16

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(label_ds)), labels)):
        print(f"\n{'='*50}")
        print(f"{task.upper()} CV Fold {fold_idx+1}/{n_folds}")
        print(f"{'='*50}")

        if task == "df":
            train_full = DFDataset(data_root, split="train", split_seed=seed)
            val_full   = DFDataset(data_root, split="train", split_seed=seed)
        else:
            train_full = SFDataset(data_root, split="train")
            val_full   = SFDataset(data_root, split="train")

        train_full.transform = get_transforms(task, is_train=True)
        val_full.transform   = get_transforms(task, is_train=False)

        train_subset = Subset(train_full, train_idx)
        val_subset   = Subset(val_full, val_idx)

        if task == "df":
            test_ds = DFDataset(data_root, split="test")
        else:
            test_ds = SFDataset(data_root, split="test")
        test_ds.transform = get_transforms(task, is_train=False)

        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,
                                  num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False,
                                  num_workers=2, pin_memory=True)
        test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                                  num_workers=2, pin_memory=True)

        for method in methods:
            print(f"  [{method}] training...")
            torch.manual_seed(seed + fold_idx * 100)
            model = build_model(task=task, mode=method)
            model.to(device)
            print(f"    Model type: {type(model).__name__}")

            # Criterion
            if method == "ce":
                class CELocal(nn.Module):
                    def forward(self, alpha, z, targets, epoch=None):
                        loss = F.nll_loss(F.log_softmax(z, dim=-1), targets)
                        return loss, {"stage": 0, "L_ce": loss.item()}
                criterion = CELocal()

            elif method == "cumulative":
                from losses.cumulative_loss import CumulativeLoss
                cum_fn = CumulativeLoss(num_classes=4)
                class CumLocal(nn.Module):
                    def forward(self, alpha, z, targets, epoch=None):
                        ol = model.ordinal_logits if hasattr(model, 'ordinal_logits') else z
                        loss = cum_fn(ol, targets)
                        return loss, {"stage": 0, "L_cum": loss.item()}
                criterion = CumLocal()

            elif method == "edl":
                from losses.edl_loss import EDLLoss
                edl_fn = EDLLoss(num_classes=4, kl_lambda=0.1)
                class EDLLocal(nn.Module):
                    def forward(self, alpha, z, targets, epoch=None):
                        loss = edl_fn(alpha, targets, epoch, epochs)
                        return loss, {"stage": 0, "L_edl": loss.item()}
                criterion = EDLLocal()

            else:  # edl_orcu
                criterion = CombinedLoss(
                    num_classes=4, lambda_orcu=0.5, lambda_kl=0.1, orcu_t=3.0,
                    orcu_lambda_reg=0.01,
                    stage_1_epochs=5, stage_2_epochs=25, total_epochs=epochs)

            # Optimizer & scheduler
            import torch.optim as optim
            from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
            bb, hd = [], []
            for n, p in model.named_parameters():
                if not p.requires_grad:
                    continue
                (bb if n.startswith("backbone") else hd).append(p)
            optimizer = optim.AdamW([{"params": bb, "lr": 1e-4}, {"params": hd, "lr": 1e-3}],
                                    weight_decay=0.05)
            ws = LinearLR(optimizer, start_factor=0.1, total_iters=5)
            cs = CosineAnnealingLR(optimizer, T_max=epochs - 5)
            scheduler = SequentialLR(optimizer, schedulers=[ws, cs], milestones=[5])

            best_val_acc, best_state = 0.0, None

            @torch.no_grad()
            def eval_cv(model, loader):
                model.eval()
                aa, zz, tt = [], [], []
                for im, tg in loader:
                    im, tg = im.to(device), tg.to(device)
                    a, z_ = model(im)
                    if a is not None:
                        aa.append(a.cpu())
                    zz.append(z_.cpu())
                    tt.append(tg.cpu())
                a = torch.cat(aa) if aa else None
                return compute_metrics(a, torch.cat(zz), torch.cat(tt))

            for epoch in range(epochs):
                model.train()
                for images, targets in train_loader:
                    images, targets = images.to(device), targets.to(device)
                    alpha, z = model(images)
                    loss, _ = criterion(alpha, z, targets, epoch)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                scheduler.step()

                if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
                    vm = eval_cv(model, val_loader)
                    if vm["acc"] > best_val_acc:
                        best_val_acc = vm["acc"]
                        best_state = copy.deepcopy(model.state_dict())

            model.load_state_dict(best_state)
            test_metrics = eval_cv(model, test_loader)
            fold_results[method].append({
                "fold": fold_idx,
                **{k: v for k, v in test_metrics.items() if isinstance(v, (float, int))}
            })
            print(f"    Acc={test_metrics['acc']:.4f} F1={test_metrics['macro_f1']:.4f} "
                  f"QWK={test_metrics['qwk']:.4f} ECE={test_metrics['ece']:.4f}")

    # Aggregate
    summary = {}
    metric_names = ["acc", "macro_f1", "qwk", "ece", "pct_unimodal", "u_ece", "auroc_u"]
    for method in methods:
        agg = {}
        for mn in metric_names:
            vals = [f[mn] for f in fold_results[method] if mn in f]
            agg[mn] = {"mean": float(np.mean(vals)), "std": float(np.std(vals)), "values": vals}
        summary[method] = agg

    # Paired t-tests
    significance = {}
    for i, m1 in enumerate(methods):
        for j, m2 in enumerate(methods):
            if i >= j:
                continue
            pair_key = f"{m1}_vs_{m2}"
            significance[pair_key] = {}
            for mn in ["acc", "macro_f1", "qwk", "ece"]:
                v1 = [f[mn] for f in fold_results[m1]]
                v2 = [f[mn] for f in fold_results[m2]]
                t_stat, p_val = stats.ttest_rel(v1, v2)
                significance[pair_key][mn] = {
                    "t_statistic": float(t_stat), "p_value": float(p_val),
                    "significant": bool(p_val < 0.05)}

    return {
        "task": task, "k": n_folds, "methods": methods,
        "fold_results": {m: [{k: v for k, v in fr.items()} for fr in fold_results[m]]
                         for m in methods},
        "summary": {m: {k: {"mean": v["mean"], "std": v["std"]}
                        for k, v in sv.items()} for m, sv in summary.items()},
        "significance": significance,
    }


# Run CV for both tasks
for task in ["df", "sf"]:
    print(f"\n{'#'*60}")
    print(f"# 5-Fold CV: {task.upper()}")
    print(f"{'#'*60}")
    cv_result = run_cv(task, DATA_ROOT,
                       methods=["ce", "cumulative", "edl", "edl_orcu"],
                       n_folds=5, epochs=80)
    with open(os.path.join(OUTPUT_DIR, f"cv_{task}.json"), "w") as f:
        json.dump(cv_result, f, indent=2)

    print(f"\n=== {task.upper()} CV Summary (mean +/- std) ===")
    for method, agg in cv_result["summary"].items():
        print(f"  {method:<14}: Acc={agg['acc']['mean']:.4f}+/-{agg['acc']['std']:.4f}  "
              f"QWK={agg['qwk']['mean']:.4f}+/-{agg['qwk']['std']:.4f}  "
              f"ECE={agg['ece']['mean']:.4f}+/-{agg['ece']['std']:.4f}")

    print(f"\n  Significance (p < 0.05):")
    for pair, tests in cv_result["significance"].items():
        sigs = [f"{m}(p={v['p_value']:.3f})" for m, v in tests.items() if v['significant']]
        if sigs:
            print(f"    {pair}: {', '.join(sigs)}")

In [ ]:
import os, json
OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"

print("\n" + "="*78)
print("FINAL RESULTS SUMMARY")
print("="*78)

all_summary = {}

# Main experiments
for fname in ["sf_all_results.json", "df_all_results.json"]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        with open(fpath) as f:
            all_summary[fname.replace(".json", "")] = json.load(f)

# CV results
for task in ["df", "sf"]:
    fpath = os.path.join(OUTPUT_DIR, f"cv_{task}.json")
    if os.path.exists(fpath):
        with open(fpath) as f:
            cv = json.load(f)
            all_summary[f"cv_{task}"] = cv.get("summary", {})
            all_summary[f"cv_{task}_significance"] = cv.get("significance", {})

# Print main table
print("\n--- Main Experiments ---")
for exp_name in ["sf_all_results", "df_all_results"]:
    if exp_name in all_summary:
        print(f"\n{exp_name}:")
        results = all_summary[exp_name]
        header = (f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} "
                  f"{'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
        print(header)
        print("-" * len(header))
        for mode, m in results.items():
            print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} "
                  f"{m.get('qwk',0):8.4f} {m.get('ece',0):8.4f} "
                  f"{m.get('u_ece',0):8.4f} {m.get('auroc_u',0):8.4f} "
                  f"{m.get('pct_unimodal',0):7.2%}")

# Print CV summary
print("\n--- Cross-Validation ---")
for task in ["df", "sf"]:
    key = f"cv_{task}"
    if key in all_summary and all_summary[key]:
        print(f"\n{task.upper()} 5-Fold CV:")
        cv_summary = all_summary[key]
        for method, agg in cv_summary.items():
            if isinstance(agg, dict) and "acc" in agg:
                print(f"  {method:<14}: Acc={agg['acc']['mean']:.4f}+/-{agg['acc']['std']:.4f}  "
                      f"QWK={agg['qwk']['mean']:.4f}+/-{agg['qwk']['std']:.4f}")

# Lambda sweep
fpath = os.path.join(OUTPUT_DIR, "lambda_sweep_v2.json")
if os.path.exists(fpath):
    with open(fpath) as f:
        all_summary["lambda_sweep_v2"] = json.load(f)
    ls = all_summary["lambda_sweep_v2"]
    print("\n--- Lambda Sweep ---")
    results = sorted(ls.get("results", []), key=lambda x: x.get("val_acc", 0), reverse=True)
    for r in results[:5]:
        print(f"  lam_orcu={r['lambda_orcu']:.2f} lam_reg={r['lambda_reg']:.4f}: "
              f"Val={r['val_acc']:.4f} Test={r['test_acc']:.4f} QWK={r['test_qwk']:.4f}")
    print(f"  Best: lam_orcu={ls['best_combo'][0]}, lam_reg={ls['best_combo'][1]}")

# Temperature sweep
fpath = os.path.join(OUTPUT_DIR, "temperature_sweep.json")
if os.path.exists(fpath):
    with open(fpath) as f:
        all_summary["temperature_sweep"] = json.load(f)
    ts = all_summary["temperature_sweep"]
    print("\n--- Temperature Sweep (DF EDL) ---")
    for task, temps in ts.items():
        best_ece = min(temps.items(), key=lambda x: x[1]["ece"])
        best_auroc = max(temps.items(), key=lambda x: x[1]["auroc_u"])
        print(f"  {task}: Best ECE={best_ece[1]['ece']:.4f} at T={best_ece[0]}  "
              f"Best AUROC(u)={best_auroc[1]['auroc_u']:.4f} at T={best_auroc[0]}")

# Save master summary
with open(os.path.join(OUTPUT_DIR, "master_summary.json"), "w") as f:
    json.dump(all_summary, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}/master_summary.json")
print("\n=== DONE ===")
print("Download the /kaggle/working/ directory for all model checkpoints and result files.")

## 7. EDL Temperature Calibration
Apply temperature scaling to EDL predictions and find optimal temperature for calibration.
Lower temperature sharpens predictions; higher temperature smooths them (reduces overconfidence).

In [ ]:
# ---- Temperature Scaling Calibration ----
import os, json, copy
import numpy as np
import torch

OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def evaluate_with_temperature(model, loader, T):
    """Evaluate model with temperature scaling on logits."""
    model.eval()
    import sys
    sys.path.insert(0, "/kaggle/working/fluorosis_project/06_Implementation")
    from eval import compute_metrics
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None:
            all_alpha.append(alpha.cpu())
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha) if all_alpha else None
    z = torch.cat(all_z)
    targets = torch.cat(all_targets)
    return compute_metrics(alpha, z, targets, temperature=T)

# Run temperature sweep on best EDL model
temperatures = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0]
print("Temperature sweep on DF EDL model...")
temp_results = {}

for task in ["df"]:  # Focus on DF where EDL performed best
    import sys
    sys.path.insert(0, "/kaggle/working/fluorosis_project/06_Implementation")
    from models import build_model
    from data import create_dataloaders
    
    # Load trained model
    model = build_model(task=task, mode="edl")
    ckpt_path = os.path.join(OUTPUT_DIR, f"{task}_edl", "best.pt")
    if not os.path.exists(ckpt_path):
        print(f"  {task}: no EDL checkpoint found, skipping")
        continue
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    
    _, _, test_loader = create_dataloaders(DATA_ROOT, task=task, batch_size=32, num_workers=2)
    temp_results[task] = {}
    for T in temperatures:
        m = evaluate_with_temperature(model, test_loader, T)
        temp_results[task][f"T_{T}"] = {
            "acc": m["acc"], "ece": m["ece"], "auroc_u": m["auroc_u"],
            "qwk": m["qwk"], "u_ece": m["u_ece"]
        }
        print(f"  T={T:.1f}: Acc={m['acc']:.4f} ECE={m['ece']:.4f} QWK={m['qwk']:.4f} AUROC(u)={m['auroc_u']:.4f}")

# Find best temperature
for task in temp_results:
    best_ece = min(temp_results[task].items(), key=lambda x: x[1]["ece"])
    best_auroc = max(temp_results[task].items(), key=lambda x: x[1]["auroc_u"])
    print(f"\n{task.upper()}: Best ECE at T={best_ece[0]} (ECE={best_ece[1]['ece']:.4f})")
    print(f"{task.upper()}: Best AUROC(u) at T={best_auroc[0]} (AUROC(u)={best_auroc[1]['auroc_u']:.4f})")

with open(os.path.join(OUTPUT_DIR, "temperature_sweep.json"), "w") as f:
    json.dump(temp_results, f, indent=2)
print("\nTemperature sweep saved.")

## 8. EDL+ORCU Lambda Sweep (DF)
Sweep lambda_orcu and lambda_reg to find optimal ORCU regularization strength.
Shorter training (50 epochs) for efficiency. Best combo will be validated with full training.

In [ ]:
# ---- EDL+ORCU Hyperparameter Sweep (DF, 50 epochs) ----
import os, sys, json, copy, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

sys.path.insert(0, "/kaggle/working/fluorosis_project/06_Implementation")
from data import create_dataloaders
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

OUTPUT_DIR = "/kaggle/working"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Sweep grid
lambda_orcu_vals = [0.1, 0.3, 0.5]
lambda_reg_vals = [0.005, 0.01, 0.02]
sweep_epochs = 50
sweep_seed = 42

print("EDL+ORCU Lambda Sweep — DF Task")
print(f"  lambda_orcu: {lambda_orcu_vals}")
print(f"  lambda_reg:  {lambda_reg_vals}")
print(f"  epochs: {sweep_epochs} per run")
print(f"  total runs: {len(lambda_orcu_vals) * len(lambda_reg_vals)}\n")

# Data (shared across sweep)
train_loader, val_loader, test_loader = create_dataloaders(
    DATA_ROOT, task="df", batch_size=32, num_workers=2)

sweep_results = []
best_val_acc = 0.0
best_combo = None

@torch.no_grad()
def eval_sweep(model, loader):
    model.eval()
    aa, zz, tt = [], [], []
    for im, tg in loader:
        im, tg = im.to(device), tg.to(device)
        a, z = model(im)
        if a is not None:
            aa.append(a.cpu())
        zz.append(z.cpu())
        tt.append(tg.cpu())
    a = torch.cat(aa) if aa else None
    return compute_metrics(a, torch.cat(zz), torch.cat(tt))

for lam_orcu, lam_reg in itertools.product(lambda_orcu_vals, lambda_reg_vals):
    print(f"  [{lam_orcu}, {lam_reg}] training...")
    torch.manual_seed(sweep_seed)
    model = build_model(task="df", mode="edl_orcu")
    model.to(device)
    
    criterion = CombinedLoss(
        num_classes=4, lambda_orcu=lam_orcu, lambda_kl=0.1, orcu_t=3.0,
        orcu_lambda_reg=lam_reg,
        stage_1_epochs=5, stage_2_epochs=15, total_epochs=sweep_epochs)
    
    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (bb if n.startswith("backbone") else hd).append(p)
    optimizer = optim.AdamW([{"params": bb, "lr": 1e-4}, {"params": hd, "lr": 1e-3}],
                            weight_decay=0.05)
    ws = LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cs = CosineAnnealingLR(optimizer, T_max=sweep_epochs - 5)
    scheduler = SequentialLR(optimizer, schedulers=[ws, cs], milestones=[5])
    
    best_val = 0.0
    best_state = None
    for epoch in range(sweep_epochs):
        model.train()
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, _ = criterion(alpha, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
        if (epoch + 1) % 10 == 0 or epoch == sweep_epochs - 1:
            vm = eval_sweep(model, val_loader)
            if vm["acc"] > best_val:
                best_val = vm["acc"]
                best_state = copy.deepcopy(model.state_dict())
    
    model.load_state_dict(best_state)
    tm = eval_sweep(model, test_loader)
    
    result = {
        "lambda_orcu": lam_orcu, "lambda_reg": lam_reg,
        "val_acc": best_val,
        "test_acc": tm["acc"], "test_qwk": tm["qwk"],
        "test_ece": tm["ece"], "test_unim": tm["pct_unimodal"],
        "test_u_ece": tm["u_ece"], "test_auroc_u": tm["auroc_u"],
    }
    sweep_results.append(result)
    
    if best_val > best_val_acc:
        best_val_acc = best_val
        best_combo = (lam_orcu, lam_reg)
    
    print(f"    Val Acc={best_val:.4f} | Test Acc={tm['acc']:.4f} QWK={tm['qwk']:.4f} "
          f"ECE={tm['ece']:.4f} Unim={tm['pct_unimodal']:.2%} AUROC(u)={tm['auroc_u']:.4f}")

# Summary table
print(f"\n{'='*80}")
print(f"Lambda Sweep Results (sorted by Val Acc)")
print(f"{'='*80}")
print(f"{'λ_orcu':>8} {'λ_reg':>8} {'Val Acc':>8} {'Test Acc':>8} {'QWK':>8} {'ECE':>8} {'Unim':>8} {'AUROC(u)':>8}")
print("-" * 72)
for r in sorted(sweep_results, key=lambda x: x["val_acc"], reverse=True):
    print(f"{r['lambda_orcu']:8.3f} {r['lambda_reg']:8.4f} "
          f"{r['val_acc']:8.4f} {r['test_acc']:8.4f} {r['test_qwk']:8.4f} "
          f"{r['test_ece']:8.4f} {r['test_unim']:7.2%} {r['test_auroc_u']:8.4f}")

print(f"\nBest combo: lambda_orcu={best_combo[0]}, lambda_reg={best_combo[1]} (Val Acc={best_val_acc:.4f})")

# Save
with open(os.path.join(OUTPUT_DIR, "lambda_sweep_v2.json"), "w") as f:
    json.dump({"results": sweep_results, "best_combo": list(best_combo)}, f, indent=2)
print("Lambda sweep saved to lambda_sweep_v2.json")

## 9. Results Summary
Compile all experiment results into a single summary file.
